In [0]:
# ═══════════════════════════════════════════
# CELL 1 — COMPLETE SETUP (ALL IN ONE)
# ═══════════════════════════════════════════
from pyspark.sql.functions import (
    current_timestamp, col, lit, upper, trim,
    to_date, to_timestamp, when,
    round as spark_round,
    sum as spark_sum, count, avg,
    max as spark_max, min as spark_min,
    countDistinct
)
from delta.tables import DeltaTable

# ─────────────────────────────────────────
# STEP 1 — STORAGE AUTH
# ─────────────────────────────────────────
storage_account = "adlssumitmod"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)
print("✅ Storage authenticated!")

# ─────────────────────────────────────────
# STEP 2 — WIDGETS
# ─────────────────────────────────────────
dbutils.widgets.text("p_date",            "2017-01-05")
dbutils.widgets.text("env",               "dev")
dbutils.widgets.text("storage_account",   "adlssumitmod")
dbutils.widgets.text("landing_container", "landing")
dbutils.widgets.text("bronze_container",  "bronze")

p_date            = dbutils.widgets.get("p_date")
env               = dbutils.widgets.get("env")
storage_account   = dbutils.widgets.get("storage_account")
landing_container = dbutils.widgets.get("landing_container")
bronze_container  = dbutils.widgets.get("bronze_container")

# ─────────────────────────────────────────
# STEP 3 — SAFETY: Fix wrong date from ADF
# ─────────────────────────────────────────
VALID_START_DATE = "2017-01-05"

if p_date < VALID_START_DATE:
    print(f"⚠️  ADF passed p_date={p_date} — no data exists!")
    print(f"⚠️  Auto-correcting to {VALID_START_DATE}")
    p_date = VALID_START_DATE

print(f"✅ Final p_date : {p_date}")

# ─────────────────────────────────────────
# STEP 4 — PARSE DATE
# ─────────────────────────────────────────
year  = p_date[:4]
month = p_date[5:7]
day   = p_date[8:10]

print(f"✅ year={year} month={month} day={day}")

# ─────────────────────────────────────────
# STEP 5 — BUILD ALL PATHS
# ─────────────────────────────────────────
base_landing = f"abfss://{landing_container}@{storage_account}.dfs.core.windows.net"
base_bronze  = f"abfss://{bronze_container}@{storage_account}.dfs.core.windows.net"
base_silver  = f"abfss://silver@{storage_account}.dfs.core.windows.net"
base_gold    = f"abfss://gold@{storage_account}.dfs.core.windows.net"

input_path   = f"{base_landing}/transactions/year={year}/month={month}/day={day}/"
bronze_path  = f"{base_bronze}/transactions"
silver_path  = f"{base_silver}/transactions"
gold_path    = f"{base_gold}"

# ─────────────────────────────────────────
# STEP 6 — VALIDATE PATH EXISTS
# ─────────────────────────────────────────
try:
    files = dbutils.fs.ls(input_path)
    print(f"✅ Path exists! {len(files)} files found:")
    for f in files:
        print(f"   → {f.name}")
except Exception as e:
    raise Exception(f"""
    ❌ PATH NOT FOUND: {input_path}
    ─────────────────────────────────
    p_date passed : {p_date}
    year          : {year}
    month         : {month}
    day           : {day}
    ─────────────────────────────────
    Valid dates start from: 2017-01-05
    Fix: Update p_date in ADF Notebook 
         Activity Base Parameters!
    """)

# ─────────────────────────────────────────
# STEP 7 — PRINT SUMMARY
# ─────────────────────────────────────────
print(f"""
✅ SETUP COMPLETE
─────────────────────────────────
p_date       : {p_date}
env          : {env}
storage      : {storage_account}
─────────────────────────────────
input_path   : {input_path}
bronze_path  : {bronze_path}
silver_path  : {silver_path}
gold_path    : {gold_path}
─────────────────────────────────
""")

# THREE MEDALLIAN ARCHITECTURE

In [0]:
storage_account_name = "adlssumitmod"
container_name = "raw"

mount_point = "/mnt/raw"
 
configs = {
  f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net": access_key
}
 
try:
    if any(mount.mountPoint == mount_point for mount in dbutils.fs.mounts()):
        print(f"{mount_point} is already mounted.")
    else:
        dbutils.fs.mount(
          source = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/",
          mount_point = mount_point,
          extra_configs = configs
        )
        print("✅ Successfully mounted ADLS Gen2!")
except Exception as e:
    print(f"❌ Mount failed. Error: {str(e)[:200]}")
 

In [0]:
storage_account_name = "adlssumitmod"
container_name = "bronze"

mount_point = "/mnt/bronze"
 
configs = {
  f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net": access_key
}
 
try:
    if any(mount.mountPoint == mount_point for mount in dbutils.fs.mounts()):
        print(f"{mount_point} is already mounted.")
    else:
        dbutils.fs.mount(
          source = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/",
          mount_point = mount_point,
          extra_configs = configs
        )
        print("✅ Successfully mounted ADLS Gen2!")
except Exception as e:
    print(f"❌ Mount failed. Error: {str(e)[:200]}")
 

In [0]:
from pyspark.sql.functions import (
    current_timestamp, col, lit, upper, trim,
    to_date, when, round as spark_round,
    sum as spark_sum, count, avg,
    max as spark_max, min as spark_min,
    countDistinct
)
from delta.tables import DeltaTable

# ─────────────────────────────────────────
# WIDGETS — match your exact setup
# ─────────────────────────────────────────
dbutils.widgets.text("env",                "dev")
dbutils.widgets.text("run_date",           "")
dbutils.widgets.text("storage_account",    "adlssumitmod")
dbutils.widgets.text("landing_container",  "landing")
dbutils.widgets.text("bronze_container",   "bronze")
dbutils.widgets.text("p_date",             "2017-01-01")

env                = dbutils.widgets.get("env")
storage_account    = dbutils.widgets.get("storage_account")
landing_container  = dbutils.widgets.get("landing_container")
bronze_container   = dbutils.widgets.get("bronze_container")
p_date             = dbutils.widgets.get("p_date")

# ─────────────────────────────────────────
# PARSE DATE
# ─────────────────────────────────────────
year  = p_date[:4]
month = p_date[5:7]
day   = p_date[8:10]

# ─────────────────────────────────────────
# BUILD CORRECT PATHS
# ─────────────────────────────────────────
base_landing = f"abfss://{landing_container}@{storage_account}.dfs.core.windows.net"
base_bronze  = f"abfss://{bronze_container}@{storage_account}.dfs.core.windows.net"

# ✅ Reads ALL csv files for that date partition
input_path  = f"{base_landing}/transactions/year={year}/month={month}/day={day}/"

bronze_path        = f"{base_bronze}/transactions"
silver_path        = f"{base_bronze}/silver/transactions"
gold_path_daily    = f"{base_bronze}/gold/transactions_daily_kpi"
gold_path_customer = f"{base_bronze}/gold/customer_summary"

print(f"✅ env              : {env}")
print(f"✅ p_date           : {p_date}")
print(f"✅ storage_account  : {storage_account}")
print(f"✅ input_path       : {input_path}")
print(f"✅ bronze_path      : {bronze_path}")

In [0]:
from pyspark.sql.functions import current_timestamp, lit

# ─────────────────────────────────────────
# FORCE CORRECT DATE — override whatever
# ADF passed
# ─────────────────────────────────────────
try:
    p_date = dbutils.widgets.get("p_date")
except:
    p_date = "2017-01-05"

# SAFETY — if date has no data, fix it
VALID_START_DATE = "2017-01-05"
if p_date < VALID_START_DATE:
    print(f"⚠️ p_date={p_date} has no data!")
    p_date = VALID_START_DATE

# REPARSE DATE PARTS
year  = p_date[:4]
month = p_date[5:7]
day   = p_date[8:10]

# REBUILD input_path WITH CORRECT DATE
input_path = f"abfss://landing@adlssumitmod.dfs.core.windows.net/transactions/year={year}/month={month}/day={day}/"

print(f"✅ p_date     : {p_date}")
print(f"✅ input_path : {input_path}")

# VERIFY FILES EXIST
files = dbutils.fs.ls(input_path)
print(f"✅ Files found: {[f.name for f in files]}")

# ─────────────────────────────────────────
# READ EACH CSV FILE SEPARATELY
# ─────────────────────────────────────────
df_orders = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{input_path}orders_{p_date}.csv")

df_order_items = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{input_path}order_items_{p_date}.csv")

df_payments = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{input_path}order_payments_{p_date}.csv")

df_reviews = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{input_path}order_reviews_{p_date}.csv")

print(f"✅ orders      : {df_orders.count()} rows")
print(f"✅ order_items : {df_order_items.count()} rows")
print(f"✅ payments    : {df_payments.count()} rows")
print(f"✅ reviews     : {df_reviews.count()} rows")

# ─────────────────────────────────────────
# ADD METADATA
# ─────────────────────────────────────────
def add_metadata(df, source_name):
    return df \
        .withColumn("ingestion_time", current_timestamp()) \
        .withColumn("source_file",    lit(source_name)) \
        .withColumn("p_date",         lit(p_date)) \
        .withColumn("env",            lit(env))

df_orders_b   = add_metadata(df_orders,      "orders")
df_items_b    = add_metadata(df_order_items, "order_items")
df_payments_b = add_metadata(df_payments,    "order_payments")
df_reviews_b  = add_metadata(df_reviews,     "order_reviews")

# ─────────────────────────────────────────
# WRITE BRONZE DELTA TABLES
# ─────────────────────────────────────────
df_orders_b.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .partitionBy("p_date") \
    .save(f"{bronze_path}/orders")

df_items_b.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .partitionBy("p_date") \
    .save(f"{bronze_path}/order_items")

df_payments_b.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .partitionBy("p_date") \
    .save(f"{bronze_path}/order_payments")

df_reviews_b.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .partitionBy("p_date") \
    .save(f"{bronze_path}/order_reviews")

print("=" * 50)
print("✅ Bronze complete!")
print(f"   → {bronze_path}/orders")
print(f"   → {bronze_path}/order_items")
print(f"   → {bronze_path}/order_payments")
print(f"   → {bronze_path}/order_reviews")
print("=" * 50)

In [0]:
# ─────────────────────────────────────────
# TEMPORARY — paste key directly for testing
# ─────────────────────────────────────────
storage_account = "adlssumitmod"

# Go to Azure Portal → Storage Account adlssumitmod
# → Security + Networking → Access Keys → Copy Key1


spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_account_key
)

print("✅ Storage authenticated!")

In [0]:
# CHECK what months exist in year=2017
check_path = f"abfss://landing@adlssumitmod.dfs.core.windows.net/transactions/year=2017/"

files = dbutils.fs.ls(check_path)
for f in files:
    print(f.path)

In [0]:
# Replace month=XX with whatever showed above
month_path = f"abfss://landing@adlssumitmod.dfs.core.windows.net/transactions/year=2017/month=01/"

days = dbutils.fs.ls(month_path)
for d in days:
    print(d.path)

In [0]:
dbutils.widgets.remove("p_date")
dbutils.widgets.text("p_date", "2017-01-05")

In [0]:
# Confirm files exist in day=05
files = dbutils.fs.ls(
    "abfss://landing@adlssumitmod.dfs.core.windows.net/transactions/year=2017/month=01/day=05/"
)
for f in files:
    print(f.name, f.size)

In [0]:
# RESET WIDGET TO CORRECT DATE
dbutils.widgets.remove("p_date")
dbutils.widgets.text("p_date", "2017-01-05")
p_date = "2017-01-05"

year  = p_date[:4]   # 2017
month = p_date[5:7]  # 01
day   = p_date[8:10] # 05

print(f"✅ p_date : {p_date}")
print(f"✅ year   : {year}")
print(f"✅ month  : {month}")
print(f"✅ day    : {day}")

In [0]:
input_path = f"abfss://landing@adlssumitmod.dfs.core.windows.net/transactions/year={year}/month={month}/day={day}/"
print(f"✅ input_path : {input_path}")

# Should print:
# abfss://landing@adlssumitmod.dfs.core.windows.net/transactions/year=2017/month=01/day=05/

In [0]:
df_orders = spark.read.format("csv") \
    .option("header","true") \
    .option("inferSchema","true") \
    .load(f"{input_path}orders_{p_date}.csv")

print(f"✅ orders : {df_orders.count()} rows")

In [0]:
# READ ALL 4 CSV FILES
df_orders = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{input_path}orders_{p_date}.csv")

df_order_items = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{input_path}order_items_{p_date}.csv")

df_payments = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{input_path}order_payments_{p_date}.csv")

df_reviews = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{input_path}order_reviews_{p_date}.csv")

print(f"✅ orders      : {df_orders.count()} rows")
print(f"✅ order_items : {df_order_items.count()} rows")
print(f"✅ payments    : {df_payments.count()} rows")
print(f"✅ reviews     : {df_reviews.count()} rows")

# ─────────────────────────────────────────
# ADD METADATA TO EACH
# ─────────────────────────────────────────
def add_metadata(df, source_name):
    return df \
        .withColumn("ingestion_time", current_timestamp()) \
        .withColumn("source_file",    lit(source_name)) \
        .withColumn("p_date",         lit(p_date)) \
        .withColumn("env",            lit(env))

df_orders_b   = add_metadata(df_orders,      "orders")
df_items_b    = add_metadata(df_order_items, "order_items")
df_payments_b = add_metadata(df_payments,    "order_payments")
df_reviews_b  = add_metadata(df_reviews,     "order_reviews")

# ─────────────────────────────────────────
# WRITE BRONZE DELTA TABLES
# ─────────────────────────────────────────
df_orders_b.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .partitionBy("p_date") \
    .save(f"{bronze_path}/orders")

df_items_b.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .partitionBy("p_date") \
    .save(f"{bronze_path}/order_items")

df_payments_b.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .partitionBy("p_date") \
    .save(f"{bronze_path}/order_payments")

df_reviews_b.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .partitionBy("p_date") \
    .save(f"{bronze_path}/order_reviews")

print("✅ Bronze layer written for all 4 tables!")
print(f"   → {bronze_path}/orders")
print(f"   → {bronze_path}/order_items")
print(f"   → {bronze_path}/order_payments")
print(f"   → {bronze_path}/order_reviews")

## SILVER LAYER

In [0]:
from pyspark.sql.functions import upper, trim, to_timestamp, col, lit, when, round as spark_round

# ─────────────────────────────────────────
# READ FROM BRONZE
# ─────────────────────────────────────────
df_orders_b   = spark.read.format("delta").load(f"{bronze_path}/orders").filter(col("p_date") == p_date)
df_items_b    = spark.read.format("delta").load(f"{bronze_path}/order_items").filter(col("p_date") == p_date)
df_payments_b = spark.read.format("delta").load(f"{bronze_path}/order_payments").filter(col("p_date") == p_date)
df_reviews_b  = spark.read.format("delta").load(f"{bronze_path}/order_reviews").filter(col("p_date") == p_date)

print(f"✅ Bronze orders read   : {df_orders_b.count()} rows")
print(f"✅ Bronze items read    : {df_items_b.count()} rows")
print(f"✅ Bronze payments read : {df_payments_b.count()} rows")
print(f"✅ Bronze reviews read  : {df_reviews_b.count()} rows")

# ─────────────────────────────────────────
# CLEAN ORDERS
# ─────────────────────────────────────────
df_orders_clean = df_orders_b \
    .dropDuplicates(["order_id"]) \
    .dropna(subset=["order_id", "customer_id"]) \
    .withColumn("order_status", upper(trim(col("order_status")))) \
    .withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp"))) \
    .withColumn("order_status_category",
        when(col("order_status") == "DELIVERED", "COMPLETED")
        .when(col("order_status") == "SHIPPED",  "IN_PROGRESS")
        .when(col("order_status") == "CANCELED", "CANCELLED")
        .otherwise("OTHER")
    ) \
    .withColumn("p_date", lit(p_date))

print(f"✅ Orders clean  : {df_orders_clean.count()} rows")

# ─────────────────────────────────────────
# CLEAN ORDER ITEMS
# ─────────────────────────────────────────
df_items_clean = df_items_b \
    .dropDuplicates(["order_id", "order_item_id"]) \
    .dropna(subset=["order_id", "product_id"]) \
    .withColumn("price",         spark_round(col("price").cast("double"), 2)) \
    .withColumn("freight_value", spark_round(col("freight_value").cast("double"), 2)) \
    .withColumn("total_item_value", spark_round(col("price") + col("freight_value"), 2)) \
    .withColumn("p_date", lit(p_date))

print(f"✅ Items clean   : {df_items_clean.count()} rows")

# ─────────────────────────────────────────
# CLEAN PAYMENTS
# ─────────────────────────────────────────
df_payments_clean = df_payments_b \
    .dropDuplicates(["order_id", "payment_sequential"]) \
    .dropna(subset=["order_id", "payment_value"]) \
    .withColumn("payment_value", spark_round(col("payment_value").cast("double"), 2)) \
    .withColumn("payment_type",  upper(trim(col("payment_type")))) \
    .withColumn("payment_category",
        when(col("payment_value") < 100,   "LOW")
        .when(col("payment_value") < 500,  "MEDIUM")
        .when(col("payment_value") < 1000, "HIGH")
        .otherwise("PREMIUM")
    ) \
    .withColumn("p_date", lit(p_date))

print(f"✅ Payments clean: {df_payments_clean.count()} rows")

# ─────────────────────────────────────────
# CLEAN REVIEWS — using rlike to safely cast
# ─────────────────────────────────────────
df_reviews_clean = df_reviews_b \
    .dropDuplicates(["review_id"]) \
    .dropna(subset=["review_id", "order_id"]) \
    .withColumn("review_score",
        when(col("review_score").rlike("^[1-5]$"),
            col("review_score").cast("integer")
        ).otherwise(None)
    ) \
    .withColumn("review_sentiment",
        when(col("review_score") >= 4, "POSITIVE")
        .when(col("review_score") == 3, "NEUTRAL")
        .when(col("review_score") <= 2, "NEGATIVE")
        .otherwise("UNKNOWN")
    ) \
    .withColumn("p_date", lit(p_date))

print(f"✅ Reviews clean : {df_reviews_clean.count()} rows")
df_reviews_clean.select("review_id", "review_score", "review_sentiment").show(5, truncate=False)

# ─────────────────────────────────────────
# DELTA MERGE UPSERT FUNCTION
# ─────────────────────────────────────────
def write_silver_merge(df, path, merge_key):
    if DeltaTable.isDeltaTable(spark, path):
        delta_table = DeltaTable.forPath(spark, path)
        delta_table.alias("target") \
            .merge(
                df.alias("source"),
                f"target.{merge_key} = source.{merge_key} AND target.p_date = source.p_date"
            ) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
        print(f"✅ UPSERTED : {path}")
    else:
        df.write.format("delta") \
            .mode("overwrite") \
            .option("mergeSchema", "true") \
            .partitionBy("p_date") \
            .save(path)
        print(f"✅ CREATED  : {path}")

# ─────────────────────────────────────────
# WRITE SILVER TABLES
# ─────────────────────────────────────────
write_silver_merge(df_orders_clean,   f"{silver_path}/orders",         "order_id")
write_silver_merge(df_items_clean,    f"{silver_path}/order_items",    "order_id")
write_silver_merge(df_payments_clean, f"{silver_path}/order_payments", "order_id")
write_silver_merge(df_reviews_clean,  f"{silver_path}/order_reviews",  "review_id")

print("=" * 50)
print("✅ Silver layer complete!")
print(f"   → {silver_path}/orders")
print(f"   → {silver_path}/order_items")
print(f"   → {silver_path}/order_payments")
print(f"   → {silver_path}/order_reviews")
print("=" * 50)

In [0]:
from pyspark.sql.functions import (
    sum as spark_sum, count, avg,
    max as spark_max, min as spark_min,
    countDistinct, round as spark_round
)

# ─────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────
gold_path_orders_kpi    = f"{base_bronze}/gold/orders_kpi"
gold_path_revenue       = f"{base_bronze}/gold/revenue_summary"
gold_path_items         = f"{base_bronze}/gold/items_summary"
gold_path_reviews       = f"{base_bronze}/gold/reviews_summary"
gold_path_master        = f"{base_bronze}/gold/master_summary"

# ─────────────────────────────────────────
# READ FROM SILVER
# ─────────────────────────────────────────
df_orders_s   = spark.read.format("delta").load(f"{silver_path}/orders")
df_items_s    = spark.read.format("delta").load(f"{silver_path}/order_items")
df_payments_s = spark.read.format("delta").load(f"{silver_path}/order_payments")
df_reviews_s  = spark.read.format("delta").load(f"{silver_path}/order_reviews")

print(f"✅ Silver orders read   : {df_orders_s.count()} rows")
print(f"✅ Silver items read    : {df_items_s.count()} rows")
print(f"✅ Silver payments read : {df_payments_s.count()} rows")
print(f"✅ Silver reviews read  : {df_reviews_s.count()} rows")

# ─────────────────────────────────────────
# KPI 1 — Daily Orders Summary
# ─────────────────────────────────────────
df_orders_kpi = df_orders_s \
    .groupBy("p_date", "order_status", "order_status_category") \
    .agg(
        count("order_id").alias("total_orders"),
        countDistinct("customer_id").alias("unique_customers")
    ) \
    .withColumn("p_date", lit(p_date))

print("✅ KPI 1 — Orders Summary:")
df_orders_kpi.show(truncate=False)

# ─────────────────────────────────────────
# KPI 2 — Revenue Summary by Payment Type
# ─────────────────────────────────────────
df_revenue_kpi = df_payments_s \
    .groupBy("p_date", "payment_type", "payment_category") \
    .agg(
        count("order_id").alias("total_transactions"),
        spark_sum("payment_value").alias("total_revenue"),
        spark_round(avg("payment_value"), 2).alias("avg_order_value"),
        spark_max("payment_value").alias("max_order_value"),
        spark_min("payment_value").alias("min_order_value")
    ) \
    .withColumn("p_date", lit(p_date))

print("✅ KPI 2 — Revenue Summary:")
df_revenue_kpi.show(truncate=False)

# ─────────────────────────────────────────
# KPI 3 — Items Summary
# ─────────────────────────────────────────
df_items_kpi = df_items_s \
    .groupBy("p_date") \
    .agg(
        count("order_id").alias("total_items_sold"),
        spark_round(spark_sum("price"), 2).alias("total_items_revenue"),
        spark_round(avg("price"), 2).alias("avg_item_price"),
        spark_round(spark_sum("freight_value"), 2).alias("total_freight"),
        spark_round(spark_sum("total_item_value"), 2).alias("total_value_incl_freight")
    ) \
    .withColumn("p_date", lit(p_date))

print("✅ KPI 3 — Items Summary:")
df_items_kpi.show(truncate=False)

# ─────────────────────────────────────────
# KPI 4 — Reviews Sentiment Summary
# ─────────────────────────────────────────
df_reviews_kpi = df_reviews_s \
    .groupBy("p_date", "review_sentiment") \
    .agg(
        count("review_id").alias("total_reviews"),
        spark_round(avg("review_score"), 2).alias("avg_review_score")
    ) \
    .withColumn("p_date", lit(p_date))

print("✅ KPI 4 — Reviews Sentiment:")
df_reviews_kpi.show(truncate=False)

# ─────────────────────────────────────────
# KPI 5 — Master Summary (JOIN ALL)
# ─────────────────────────────────────────
df_orders_total = df_orders_s \
    .groupBy("p_date") \
    .agg(count("order_id").alias("total_orders"))

df_revenue_total = df_payments_s \
    .groupBy("p_date") \
    .agg(spark_round(spark_sum("payment_value"), 2).alias("total_revenue"))

df_items_total = df_items_s \
    .groupBy("p_date") \
    .agg(count("order_id").alias("total_items"))

df_reviews_total = df_reviews_s \
    .groupBy("p_date") \
    .agg(spark_round(avg("review_score"), 2).alias("avg_review_score"))

df_master = df_orders_total \
    .join(df_revenue_total, "p_date", "left") \
    .join(df_items_total,   "p_date", "left") \
    .join(df_reviews_total, "p_date", "left")

print("✅ KPI 5 — Master Summary:")
df_master.show(truncate=False)

# ─────────────────────────────────────────
# WRITE ALL GOLD TABLES
# ─────────────────────────────────────────
df_orders_kpi.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .partitionBy("p_date") \
    .save(gold_path_orders_kpi)

df_revenue_kpi.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .partitionBy("p_date") \
    .save(gold_path_revenue)

df_items_kpi.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .partitionBy("p_date") \
    .save(gold_path_items)

df_reviews_kpi.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .partitionBy("p_date") \
    .save(gold_path_reviews)

df_master.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .partitionBy("p_date") \
    .save(gold_path_master)

print("=" * 50)
print("✅ Gold layer complete!")
print(f"   → {gold_path_orders_kpi}")
print(f"   → {gold_path_revenue}")
print(f"   → {gold_path_items}")
print(f"   → {gold_path_reviews}")
print(f"   → {gold_path_master}")
print("=" * 50)

In [0]:
# CHECK YOUR CONTAINERS
containers = ["bronze", "silver", "gold", "raw"]
for c in containers:
    try:
        path = f"abfss://{c}@{storage_account}.dfs.core.windows.net/"
        dbutils.fs.ls(path)
        print(f"✅ Container EXISTS : {c}")
    except:
        print(f"❌ Container MISSING: {c}")

In [0]:
# ─────────────────────────────────────────
# CORRECT PATHS PER CONTAINER
# ─────────────────────────────────────────
storage_account   = "adlssumitmod"
bronze_container  = "bronze"
silver_container  = "silver"
gold_container    = "gold"

base_bronze = f"abfss://{bronze_container}@{storage_account}.dfs.core.windows.net"
base_silver = f"abfss://{silver_container}@{storage_account}.dfs.core.windows.net"
base_gold   = f"abfss://{gold_container}@{storage_account}.dfs.core.windows.net"

# SILVER PATHS
silver_orders    = f"{base_silver}/transactions/orders"
silver_items     = f"{base_silver}/transactions/order_items"
silver_payments  = f"{base_silver}/transactions/order_payments"
silver_reviews   = f"{base_silver}/transactions/order_reviews"

# GOLD PATHS
gold_orders_kpi  = f"{base_gold}/orders_kpi"
gold_revenue     = f"{base_gold}/revenue_summary"
gold_items       = f"{base_gold}/items_summary"
gold_reviews     = f"{base_gold}/reviews_summary"
gold_master      = f"{base_gold}/master_summary"

print("✅ Paths configured!")
print(f"   Bronze : {base_bronze}")
print(f"   Silver : {base_silver}")
print(f"   Gold   : {base_gold}")

# ─────────────────────────────────────────
# WRITE SILVER
# ─────────────────────────────────────────
df_orders_clean.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .partitionBy("p_date") \
    .save(silver_orders)
print(f"✅ Silver orders    → {silver_orders}")

df_items_clean.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .partitionBy("p_date") \
    .save(silver_items)
print(f"✅ Silver items     → {silver_items}")

df_payments_clean.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .partitionBy("p_date") \
    .save(silver_payments)
print(f"✅ Silver payments  → {silver_payments}")

df_reviews_clean.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .partitionBy("p_date") \
    .save(silver_reviews)
print(f"✅ Silver reviews   → {silver_reviews}")

# ─────────────────────────────────────────
# WRITE GOLD
# ─────────────────────────────────────────
df_orders_kpi.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .partitionBy("p_date") \
    .save(gold_orders_kpi)
print(f"✅ Gold orders_kpi  → {gold_orders_kpi}")

df_revenue_kpi.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .partitionBy("p_date") \
    .save(gold_revenue)
print(f"✅ Gold revenue     → {gold_revenue}")

df_items_kpi.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .partitionBy("p_date") \
    .save(gold_items)
print(f"✅ Gold items       → {gold_items}")

df_reviews_kpi.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .partitionBy("p_date") \
    .save(gold_reviews)
print(f"✅ Gold reviews     → {gold_reviews}")

df_master.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .partitionBy("p_date") \
    .save(gold_master)
print(f"✅ Gold master      → {gold_master}")

print("=" * 50)
print("🏆 ALL LAYERS WRITTEN SUCCESSFULLY!")
print("=" * 50)

# OPTIMIZATION REPORTS



In [0]:
import time
from pyspark.sql.functions import col, count, sum as spark_sum, avg, round as spark_round

# ─────────────────────────────────────────
# PATHS — use your existing silver tables
# ─────────────────────────────────────────
storage_account = "adlssumitmod"

silver_orders   = f"abfss://silver@{storage_account}.dfs.core.windows.net/transactions/orders"
silver_payments = f"abfss://silver@{storage_account}.dfs.core.windows.net/transactions/order_payments"
silver_items    = f"abfss://silver@{storage_account}.dfs.core.windows.net/transactions/order_items"

print("✅ Paths ready!")

In [0]:
# Analyze execution plan BEFORE optimization
print("📊 BEFORE OPTIMIZATION - Execution Plan")
before_opt_start = time.time()

# Sample query to measure
sample_query_df = spark.read.format("delta").load(silver_orders) \
    .filter(col("order_status") == "DELIVERED") \
    .filter(col("order_status_category") == "COMPLETED") \
    .select("order_id", "customer_id", "order_purchase_timestamp")

# Show execution plan
sample_query_df.explain(mode="extended")

# Execute and time
sample_query_df.collect()
before_opt_end = time.time()
before_optimization_runtime = before_opt_end - before_opt_start
print(f"⏱️ Before Optimization Runtime: {before_optimization_runtime:.4f} seconds")

In [0]:
# Run OPTIMIZE on Silver table
print("🔧 Running OPTIMIZE on Silver table...")
optimize_start = time.time()

spark.sql(f"OPTIMIZE delta.`{silver_orders}`")

optimize_end = time.time()
print(f"✅ OPTIMIZE completed in {optimize_end - optimize_start:.2f} seconds")

In [0]:
# Apply Z-Ordering on frequently queried columns
print("🔧 Applying Z-Ordering on Silver table (order_status, customer_id, order_purchase_timestamp)...")
zorder_start = time.time()

spark.sql(f"""
    OPTIMIZE delta.`{silver_orders}` 
    ZORDER BY (order_status, customer_id, order_purchase_timestamp)
""")

zorder_end = time.time()
print(f"✅ Z-Ordering completed in {zorder_end - zorder_start:.2f} seconds")

In [0]:
# Cache frequently accessed data
#caching


print("🔧 Caching Silver table...")
cache_start = time.time()

silver_cached_df = spark.read.format("delta").load(silver_orders)
silver_cached_df.cache()
silver_cached_df.count()  # Trigger cache

cache_end = time.time()
print(f"✅ Cache completed in {cache_end - cache_start:.2f} seconds")

In [0]:
# Analyze execution plan AFTER optimization
print("📊 AFTER OPTIMIZATION - Execution Plan")
after_opt_start = time.time()

# Same query after optimization
optimized_query_df = silver_cached_df \
    .filter(col("order_status") == "DELIVERED") \
    .filter(col("order_status_category") == "COMPLETED") \
    .select("order_id", "customer_id", "order_purchase_timestamp")

# Show execution plan
optimized_query_df.explain(mode="extended")

# Execute and time
optimized_query_df.collect()
after_opt_end = time.time()
after_optimization_runtime = after_opt_end - after_opt_start
print(f"⏱️ After Optimization Runtime: {after_optimization_runtime:.4f} seconds")